   # Sentiment Analysis with Natural Language Processing

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sujith\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Sujith\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sujith\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## 2. Load Dataset

In [141]:
df = pd.read_csv("User Reviews.csv")

In [147]:
df.head()

,ID,Translated_Review,Sentiment
0,1,i like eat delicious food thats im cooking foo...,Positive
1,2,this help eating healthy exercise regular basis,Positive
3,4,works great especially going grocery store,Positive
4,5,best idea us,Positive
5,6,best way,Positive


## 3. Preprocess the Dataset

In [151]:
# One-hot encode Sentiment
df['Sentiment_Positive'] = (df['Sentiment'] == 'Positive').astype(int)
df['Sentiment_Neutral'] = (df['Sentiment'] == 'Neutral').astype(int)
df['Sentiment_Negative'] = (df['Sentiment'] == 'Negative').astype(int)

# Drop original Sentiment column
df.drop(columns=['Sentiment'], inplace=True)

# Use only Positive and Negative for training
df = df[df['Sentiment_Neutral'] == 0]  # Remove Neutral reviews
df.drop(columns=['Sentiment_Neutral'], inplace=True)  # Drop neutral column

# Rename target column for clarity
df.rename(columns={'Sentiment_Positive': 'Sentiment'}, inplace=True)

# Verify no NaN values
assert df['Sentiment'].isna().sum() == 0, "NaN values found in target variable"

# Prepare X and y for model training
X = df['Translated_Review']
y = df['Sentiment']

In [153]:
print(df['Sentiment'].unique())

[1 0]


In [155]:
df.head()

,ID,Translated_Review,Sentiment,Sentiment_Negative
0,1,i like eat delicious food thats im cooking foo...,1,0
1,2,this help eating healthy exercise regular basis,1,0
3,4,works great especially going grocery store,1,0
4,5,best idea us,1,0
5,6,best way,1,0


In [157]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Preprocessing function
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    tokens = word_tokenize(text)  # Tokenize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]  # Lemmatize and remove stopwords
    return ' '.join(tokens)

# Apply preprocessing
df['Translated_Review'] = df['Translated_Review'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sujith\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Sujith\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sujith\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [169]:
df.head(10)

,ID,Translated_Review,Sentiment,Sentiment_Negative
0,1,like eat delicious food thats im cooking food ...,1,0
1,2,help eating healthy exercise regular basis,1,0
3,4,work great especially going grocery store,1,0
4,5,best idea u,1,0
5,6,best way,1,0
6,7,amazing,1,0
10,11,good,1,0
11,12,useful information amount spelling error quest...,1,0
12,13,thank great app add arthritis eye immunity kid...,1,0
13,14,greatest ever completely awesome maintain heal...,1,0


## 4. Text Vectorization using TF-IDF

In [161]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorier
tfidf_vectorizer = TfidfVectorizer()

# Transform text data into numerical format
X_tfidf = tfidf_vectorizer.fit_transform(df['Translated_Review'])

In [163]:
df.head()

,ID,Translated_Review,Sentiment,Sentiment_Negative
0,1,like eat delicious food thats im cooking food ...,1,0
1,2,help eating healthy exercise regular basis,1,0
3,4,work great especially going grocery store,1,0
4,5,best idea u,1,0
5,6,best way,1,0


## 5. Model Training using Logistic Regression

In [165]:
print(df['Sentiment'].unique())

[1 0]


In [167]:
print(df['Sentiment'].isna().sum())  # Should print 0

0


In [171]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Define features (X) and target (y)
X = X_tfidf  # Already vectorized text
y = df['Sentiment']  # Target labels

# Split into train and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Logistic Regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9225
Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.76      0.83      1654
           1       0.92      0.98      0.95      4800

    accuracy                           0.92      6454
   macro avg       0.92      0.87      0.89      6454
weighted avg       0.92      0.92      0.92      6454



## 6. Model Evaluation

In [173]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Predict on test data
y_pred = model.predict(X_test)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")

Accuracy: 0.92
Precision: 0.92
Recall: 0.98
F1-Score: 0.95


In [175]:
def predict_sentiment(review):
    # Preprocess the review (same steps as during training)
    processed_review = preprocess_text(review)
    
    # Convert to TF-IDF representation
    review_tfidf = tfidf_vectorizer.transform([processed_review])
    
    # Predict sentiment
    prediction = model.predict(review_tfidf)
    
    # Convert numerical label to sentiment
    sentiment_label = "Positive" if prediction[0] == 1 else "Negative"
    
    return sentiment_label

# Example usage:
new_review = "This app is amazing and very helpful!"
print("Predicted Sentiment:", predict_sentiment(new_review))

Predicted Sentiment: Positive


In [177]:
predict_sentiment('Not that bad, but need improvements')

'Negative'

In [179]:
predict_sentiment('I have imporved a lot by this')

'Positive'

# **Conclusion: Sentiment Analysis with NLP**

## **Project Summary**  
In this project, we developed a **Sentiment Analysis** model using **Natural Language Processing (NLP)** techniques to classify customer reviews as **positive** or **negative**. We processed a dataset containing **30,000 reviews**, performed **text preprocessing**, and trained a **Logistic Regression model** on vectorized text data.

## **Key Steps and Methodology**  
### **1. Data Preprocessing**  
- **Cleaning Text:** Converted text to lowercase, removed punctuation, and tokenized words.  
- **Stopword Removal & Lemmatization:** Used NLTK to remove common stopwords and applied lemmatization for word normalization.  

### **2. Feature Engineering**  
- **TF-IDF Vectorization:** Transformed textual data into a numerical format using the **TF-IDF (Term Frequency-Inverse Document Frequency)** method.  

### **3. Model Training & Evaluation**  
- **Algorithm Used:** Logistic Regression  
- **Performance Metrics:**  
  - **Accuracy:** 92%  
  - **Precision:** 92%  
  - **Recall:** 98%  
  - **F1-Score:** 95%  

These results indicate a **highly effective model**, with excellent precision and recall, making it reliable for real-world sentiment classification.  

## **Key Takeaways & Insights**  
1. **Data Quality Matters:** Preprocessing plays a crucial role in improving model performance. Removing noise and irrelevant words enhances accuracy.  
2. **TF-IDF is Effective:** TF-IDF proved to be a powerful technique for text representation, ensuring meaningful features were extracted.  
3. **Model Simplicity & Accuracy:** Despite its simplicity, Logistic Regression achieved high performance, proving to be a strong baseline model for sentiment classification.  
4. **Scalability:** The model can be extended to classify other sentiments like **neutral**, or even fine-tuned with deep learning models (e.g., LSTMs, Transformers).  

## **Future Enhancements**  
To improve the model further, we can explore:  
✅ **Expanding the Dataset:** Incorporating more diverse reviews for better generalization.  
✅ **Using Deep Learning:** Implementing **LSTMs or BERT** to handle complex linguistic patterns.  
✅ **Aspect-Based Sentiment Analysis:** Extracting insights beyond polarity (e.g., identifying emotions like joy, anger, or frustration).  
✅ **Real-Time Deployment:** Deploying the model using **Flask, FastAPI, or a cloud-based service** for live sentiment classification.  

## **Final Thoughts**  
This project successfully demonstrates how **NLP and Machine Learning** can be leveraged to analyze customer sentiment. By implementing advanced preprocessing techniques and feature extraction, we built a highly accurate sentiment classification model. Future improvements can make the system even more robust, providing deeper insights into customer feedback. 🚀  
